In [ ]:
# MODEL: S ->A -> (R = 2) G or S->B->(R = 1) G

import random
from collections import defaultdict

random.seed(0)

gamma = 0.9
alpha = 0.5

states = ["S", "A", "B", "G"]
actions = {
    "S": ["A", "B"],   # from S you choose A or B
    "A": ["G"],        # A goes to G
    "B": ["G"],        # B goes to G
    "G": []
}

def step(state, action = None):
    """
    Deterministic transitions.
    Rewards:
      S -> A : 0
      S -> B : 0
      A -> G : 2
      B -> G : 1
    """
    if state == "S":
        next_state = action
        reward = 0
        return next_state, reward

    if state == "A":
        return "G", 2

    if state == "B":
        return "G", 1
    
    return "G", 0 # if we are in G, we stay in G and get 0 reward



In [4]:
def td0_random_policy(alpha = 0.5, gamma = 0.9, episodes=20):
    V = defaultdict(float)
    V["G"] = 0.0

    for _ in range(episodes):
        s = "S"
        while s != "G":
            if s == "S":
                 a = random.choice(actions[s])
            else:
                 a = actions[s][0] # only one action available
            s_next, reward = step(s, a)

            # TD(0) update
            V[s] = V[s] + alpha * (reward + gamma * V[s_next] - V[s])
            s = s_next
            
    return V

V = td0_random_policy(alpha = 0.5, gamma = 0.9, episodes=50)

print("TD(0) state values under random policy:")
print("V(S) =", round(V["S"], 3))
print("V(A) =", round(V["A"], 3))
print("V(B) =", round(V["B"], 3))
print("V(G) =", round(V["G"], 3))
print()


TD(0) state values under random policy:
V(S) = 1.421
V(A) = 2.0
V(B) = 1.0
V(G) = 0.0



In [9]:
# with eps probability take a random action (exploration), otherwise take the greedy
# action according to current value estimates (exploitation)
def epsilon_greedy_policy(state, V, gamma = 0.9, eps = 0.5):
    if state != "S":
        return "G"
    
    actions = ["A", "B"]
    
    # Greedy choice using current value estimates
    def action_value(a):
        next_state, reward = step("S", a)
        return reward + gamma * V[next_state]

    if random.random() < eps:
        return random.choice(actions)
    else:
        return max(actions, key = action_value) 
        # the action that leads to the state with the highest value

def td0_train_epsilon_greedy(alpha=0.5, gamma=0.9, eps=0.5, episodes=20):
    V = defaultdict(float)
    V["G"] = 0.0

    for ep in range(episodes):
        s = "S"
        while s != "G":
            a = epsilon_greedy_policy(s, V, gamma, eps) # a = next action(state)
            s_next, reward = step(s, a)

            # TD(0) update
            V[s] = V[s] + alpha * (reward + gamma * V[s_next] - V[s])
            s = s_next

    return V

V_eps = td0_train_epsilon_greedy(alpha=0.5, gamma=0.9, eps = 0.3, episodes=20)

print("\nTD with epsilon-greedy policy")
print("V(S) =", round(V_eps["S"], 3))
print("V(A) =", round(V_eps["A"], 3))
print("V(B) =", round(V_eps["B"], 3))
print("V(G) =", round(V_eps["G"], 3))


TD with epsilon-greedy policy
V(S) = 1.575
V(A) = 2.0
V(B) = 0.938
V(G) = 0.0


In [ ]:
def epsilon_greedy_Q_policy(Q, s, gamma, eps):
    if s != "S":
        return "G"
    
    actions = ["A", "B"]
    
    def action_value(a):
        next_state, reward = step("S", a)
        return reward + gamma * max(Q[(next_state, a_next)] for a_next in actions)

    if random.random() < eps:
        return random.choice(actions)
    else:
        return max(actions, key = action_value)

# ---------------------------------------------------------
# 2) SARSA: on-policy control
# ---------------------------------------------------------
def sarsa(alpha = 0.5, gamma = 0.9, eps = 0.5, episodes=50):
    Q = defaultdict(float)

    for _ in range(episodes):
        s = "S"
        a = epsilon_greedy_Q_policy(Q, s, gamma, eps)

        while s != "G":
            s_next, reward = step(s, a)

            if s_next == "G":
                target = reward
                Q[(s, a)] = Q[(s, a)] + alpha * (target - Q[(s, a)])
                break
            else:
                a_next = epsilon_greedy_policy(Q, s_next, gamma, eps)
                target = reward + gamma * Q[(s_next, a_next)]
                Q[(s, a)] = Q[(s, a)] + alpha * (target - Q[(s, a)])
                s, a = s_next, a_next

    return Q


Q_sarsa = sarsa(alpha = 0.5, gamma = 0.9, eps = 0.1, episodes = 20)
print("SARSA action values:")
for sa in Q_sarsa:
    if Q_sarsa[sa] != 0:
        print(f"Q{sa} = {round(Q_sarsa[sa], 3)}")
print()

best_sarsa = max(["A", "B"], key=lambda a: Q_sarsa[("S", a)])
print("SARSA best action from S:", best_sarsa)

SARSA action values:
Q('B', 'G') = 0.75
Q('S', 'B') = 0.225
Q('A', 'G') = 1.992
Q('S', 'A') = 1.737

SARSA best action from S: A


In [19]:
# ---------------------------------------------------------
# 3) Q-learning: off-policy control
# ---------------------------------------------------------
def q_learning(alpha = 0.5, gamma = 0.9, eps = 0.5, episodes=50):
    Q = defaultdict(float)

    for _ in range(episodes):
        s = "S"

        while s != "G":
            a = epsilon_greedy_Q_policy(Q, s, gamma, eps)
            s_next, reward = step(s, a)

            if s_next == "G":
                target = reward
            else:
                target = reward + gamma * max(Q[(s_next, a_next)] for a_next in actions[s_next])

            Q[(s, a)] = Q[(s, a)] + alpha * (target - Q[(s, a)])
            s = s_next

    return Q

Q_q = q_learning(alpha = 0.5, gamma = 0.9, eps = 0.5, episodes = 20)
for sa in Q_q:
    if Q_q[sa] != 0:
        print(f"Q{sa} = {round(Q_q[sa], 3)}")
print()

best_q = max(["A", "B"], key=lambda a: Q_q[("S", a)])

print("Q-learning best action from S:", best_q)

Q('A', 'G') = 2.0
Q('S', 'A') = 1.8
Q('B', 'G') = 0.938
Q('S', 'B') = 0.619

Q-learning best action from S: A
